# NB2 — Core-7 V2 Drop, Clean Positives & Item Metadata

NB2 dùng mapping **`core7-v2`**. Mapping V1 được giữ nguyên/frozen; V2 chỉ thêm version mới và không ghi đè artifact V1.

Output nằm trong `core7_drop_v2/` theo runtime path config.

In [ ]:
%pip install -q "datasets>=3,<5"

## 1. Runtime portable

Notebook không phụ thuộc Google Drive. Mở notebook từ repo đã clone, set `FASHION_PROJECT_ROOT`, hoặc bật `AUTO_CLONE_REPO=True`.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False

def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")
    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None

REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
if REPO_ROOT is None:
    raise RuntimeError("Không tìm thấy repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths
from src.data.prepare_core7_dataset import CORE_CATEGORIES, CORE7_ITEM_METADATA_VERSION
from src.data.prepare_core7_dataset_v2 import (
    load_category_mapping_v2,
    prepare_clean_positive_split_v2,
)

RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)
print("Repo root      :", REPO_ROOT)
print("Artifact root  :", RUNTIME_PATHS.artifact_root)
print("Core-7 output  :", RUNTIME_PATHS.core7_dir)


## 2. Mapping V2

`category_mapping_core7_v1.json` vẫn immutable. V2 kế thừa V1 và ghi rõ các override mới trong `category_mapping_core7_v2.json`.

In [ ]:
CORE7_OUTPUT_DIR = RUNTIME_PATHS.core7_dir
CORE7_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAPPING_PATH = REPO_ROOT / "configs/category_mapping_core7_v2.json"
MIN_ITEMS = 3

mapping_metadata, category_mapping = load_category_mapping_v2(MAPPING_PATH)
print("Mapping version:", mapping_metadata["mapping_version"])
print("Mapping status :", mapping_metadata["status"])
print("Base mapping   :", mapping_metadata["base_mapping_version"])
print("Categories     :", len(category_mapping))
print("Core types     :", CORE_CATEGORIES)
print("Metadata schema:", CORE7_ITEM_METADATA_VERSION)


## 3. Debug 50 train outfits

In [ ]:
DEBUG_POSITIVE_OUTPUT = CORE7_OUTPUT_DIR / "debug_category_clean_train.jsonl"
DEBUG_METADATA_OUTPUT = CORE7_OUTPUT_DIR / "debug_core7_item_metadata_v1_train.jsonl"

debug_report = prepare_clean_positive_split_v2(
    split="train",
    output_path=DEBUG_POSITIVE_OUTPUT,
    item_metadata_output_path=DEBUG_METADATA_OUTPUT,
    mapping_path=MAPPING_PATH,
    min_items=MIN_ITEMS,
    debug_limit=50,
)

summary = {
    "mapping_version": debug_report["mapping_version"],
    "positive_validation_pass": debug_report["validation"]["pass"],
    "metadata_validation_pass": debug_report["item_metadata_validation"]["pass"],
    "outfits_processed": debug_report["outfits"]["kits_processed"],
    "outfits_kept": debug_report["outfits"]["outfits_kept"],
    "metadata_item_count": debug_report["item_metadata_validation"]["item_count"],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 4. Full run

Chỉ bật sau khi debug PASS. Full output được ghi vào `core7_drop_v2/`, không đụng artifact V1.

In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_reports = {}
    for split in ("train", "valid", "test"):
        positive_path = CORE7_OUTPUT_DIR / f"category_clean_{split}.jsonl"
        metadata_path = CORE7_OUTPUT_DIR / f"core7_item_metadata_v1_{split}.jsonl"
        report_path = CORE7_OUTPUT_DIR / f"category_clean_{split}_report.json"
        report = prepare_clean_positive_split_v2(
            split=split,
            output_path=positive_path,
            item_metadata_output_path=metadata_path,
            mapping_path=MAPPING_PATH,
            min_items=MIN_ITEMS,
            debug_limit=None,
        )
        with report_path.open("w", encoding="utf-8") as stream:
            json.dump(report, stream, ensure_ascii=False, indent=2)
            stream.write("\n")
        full_reports[split] = report
    print("FULL CORE-7 V2 RUN COMPLETE")
else:
    print("RUN_FULL=False — chỉ chạy debug.")


## 5. Gate trước NB3

Chỉ sang NB3 khi cả positive validation và metadata validation đều PASS. Metadata rows phải ghi `category_mapping_version = core7-v2`.